<a href="https://colab.research.google.com/github/Subho7439/emoji_prediction/blob/main/emoji.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import pandas as pd
data = pd.read_csv("emoji_data.csv")
print(data.head())


           French macaroon is so tasty   4
0                     work is horrible   3
1                           I am upset  3 
2                       throw the ball  1 
3                            Good joke   2
4  what is your favorite baseball game   1


In [45]:
import emoji

emoji_dict = {
    0 : ":red_heart:",
    1 : ":baseball:",
    2 : ":grinning_face_with_big_eyes:",
    3 : ":disappointed_face:",
    4 : ":fork_and_knife_with_plate:",
}
def label_to_emoji(label):
  return emoji.emojize(emoji_dict[label])

# Clean the '4' column: strip spaces and convert to numeric, coercing errors
cleaned_labels = pd.to_numeric(data["4"].astype(str).str.strip(), errors='coerce')

# Filter out rows where the label could not be converted to a number (e.g., '0v2')
valid_indices = cleaned_labels.notna()
data_filtered = data[valid_indices].copy() # Use .copy() to avoid SettingWithCopyWarning
y_cleaned = cleaned_labels[valid_indices].astype(int)

# Update x and y using the filtered data
x = data_filtered["French macaroon is so tasty"].values
y = y_cleaned.values

In [18]:
import numpy as np
file = open("glove.6B.100d[1].txt",encoding="utf8")
content = file.readlines()
file.close()
embedding = {}
for line in content:
  line = line.split()
  embedding[line[0]] = np.array(line[1:],dtype = float)

In [23]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(x)
word2index = tokenizer.word_index
xtoken = tokenizer.texts_to_sequences(x)

In [34]:
def max_len(data):
  maxlen = 0
  for i in data:
    maxlen = max(maxlen,len(i))
  return maxlen

maxlen = max_len(xtoken)


In [37]:
from keras.preprocessing.sequence import pad_sequences
x_train = pad_sequences(xtoken,maxlen=maxlen,padding="post",truncating="post")

from tensorflow.keras.utils import to_categorical
y_train = to_categorical(y)

In [40]:
from keras.models import Sequential
from keras.layers import Embedding,LSTM,Dense
embed_size = 100
embedding_matrix = np.zeros((len(word2index)+1,embed_size))
for word_token, index_val in word2index.items(): # 'word_token' is the actual word, 'index_val' is its integer index
  if word_token in embedding: # Check if the word exists in the GloVe embeddings
    embed_vector = embedding[word_token]
    embedding_matrix[index_val] = embed_vector
  # If word_token not in embedding, the corresponding row in embedding_matrix remains zeros (default)

model = Sequential()
model.add(Embedding(input_dim=len(word2index)+1,output_dim=embed_size,input_length=maxlen,weights=[embedding_matrix],trainable=False))
model.add(LSTM(units=16,return_sequences=True))
model.add(LSTM(units=4))
model.add(Dense(units=5,activation="softmax"))

model.compile(optimizer="adam",loss="categorical_crossentropy",metrics=["accuracy"])

# Temporary workaround: Truncate x_train to match y_train's length if they are inconsistent
# The root cause of inconsistent data should be addressed in upstream cells.
if len(x_train) != len(y_train):
    print(f"Warning: x_train length ({len(x_train)}) does not match y_train length ({len(y_train)}). Truncating x_train.")
    x_train = x_train[:len(y_train)]

model.fit(x_train,y_train,epochs=100)

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


6/6 ━━━━━━━━━━━━━━━━━━━━ 4s 14ms/step - accuracy: 0.2044 - loss: 1.6096
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.3094 - loss: 1.5730
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3094 - loss: 1.5538
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3094 - loss: 1.5417
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3481 - loss: 1.5352
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3591 - loss: 1.5295
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3591 - loss: 1.5259
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3591 - loss: 1.5212
Epoch 9/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3536 - loss: 1.5171
Epoch 10/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3481 - loss: 1.5125
Epoch 11/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - accuracy: 0.3425 - loss: 1.5080
Epoch 12/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.3646 - loss: 1.5006
E

In [46]:
test=["I feel good","I feel very bad"]
test_token = tokenizer.texts_to_sequences(test)
x_test = pad_sequences(test_token,maxlen=maxlen,padding="post",truncating="post")
pred = model.predict(x_test)
siuu = np.argmax(pred,axis = 1)
for i in range(len(test)):
  print(test[i],label_to_emoji(siuu[i]))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step
I feel good 😞
I feel very bad 😃
